# Equação de Poisson 3D em Eletrostática — Resolução Método Galerkin-Colocação

## 1. A Equação de Poisson em Eletrostática

### 1.1 Formulação física

Em eletrostática, o potencial elétrico $\phi(\mathbf{x})$ gerado por uma distribuição de carga $\rho(\mathbf{x})$ satisfaz a equação de Poisson:

$$
\boxed{\nabla^2 \phi(\mathbf{x}) = -\frac{\rho(\mathbf{x})}{\varepsilon_0}}
$$

onde:
- $\phi(\mathbf{x})$ é o **potencial elétrico** (em Volts)
- $\rho(\mathbf{x})$ é a **densidade volumétrica de carga** (em C/m³)
- $\varepsilon_0 \approx 8.854 \times 10^{-12}$ F/m é a **permissividade do vácuo**

### 1.2 Interpretação física

- **Linearidade**: o potencial é linear na densidade de carga, pelo princípio da superposição.
- **Ausência de singularidade no interior**: no interior do domínio, $\phi$ é suave se $\rho$ for suave. Singularidades só aparecem quando $\rho$ é uma delta de Dirac (carga puntual).
- **Domínio fixo**: consideramos $\Omega = [a_x, b_x] \times [a_y, b_y] \times [a_z, b_z]$, com $a_x = a_y = a_z = -1$ e $b_x = b_y = b_z = 1$ neste desenvolvimento.
- **Condições de contorno**: nas faces do domínio, o potencial satisfaz as condições de contorno de Dirichlet, por exemplo, as faces são condutores aterrados ($\phi = 0$) ou mantidos em um potencial fixo.

### 1.3 Significado da fonte

Seja $\nabla^2 u = f$ com

$$
f(\mathbf{x}) = -\frac{\rho(\mathbf{x})}{\varepsilon_0}.
$$

A escolha que reproduz o caso do notebook corresponde a uma densidade de carga proporcional a uma autofunção do Laplaciano no cubo $[-1,1]^3$, que se anula em todas as faces.

## 2. Dedução Analítica por Separação de Variáveis

### 2.1 A equação

Resolvemos

$$
\nabla^2 \phi(x,y,z) = -\frac{\rho(x,y,z)}{\varepsilon_0}, \qquad (x,y,z) \in \Omega = [-1,1]^3,
$$

com condições de contorno de Dirichlet homogêneas:

$$
\phi = 0 \quad \text{em } \partial\Omega \ \text{(faces aterradas)}.
$$

### 2.2 Autofunções do Laplaciano no cubo $[-1,1]^3$

As autofunções do operador $-\nabla^2$ no cubo $[-1,1]^3$ com condições de Dirichlet homogêneas são:

$$
\boxed{
\psi_{nml}(x,y,z) = \sin\!\left(\frac{n\pi(x+1)}{2}\right)
\sin\!\left(\frac{m\pi(y+1)}{2}\right)
\sin\!\left(\frac{l\pi(z+1)}{2}\right),
\qquad n,m,l \in \mathbb{N}^*
}
$$

com autovalores

$$
\boxed{
\Lambda_{nml} = \frac{\pi^2}{4}(n^2 + m^2 + l^2)
}
$$

Essas funções formam uma base completa de $L^2(\Omega)$, ou seja, qualquer função suave que se anula nas faces pode ser expandida nessa base. Note que a normalização por $(x+1)/2$ mapeia $[-1,1]$ para $[0,1]$, e a autofunção se anula em $x = \pm 1$, $y = \pm 1$, $z = \pm 1$.

### 2.3 Expansão do potencial e da carga

Expandimos o potencial e a densidade de carga:

$$
\phi = \sum_{n,m,l} \Phi_{nml}\,\psi_{nml},
\qquad
\rho = \sum_{n,m,l} R_{nml}\,\psi_{nml},
$$

onde os coeficientes da carga são as projeções

$$
R_{nml} = \int_{-1}^{1}\!\!\int_{-1}^{1}\!\!\int_{-1}^{1}
\rho(x,y,z)\,\psi_{nml}(x,y,z)\,dx\,dy\,dz
\;\Big/\; \|\psi_{nml}\|^2,
$$

sendo $\|\psi_{nml}\|^2 = 1$ para a base ortonormalizada (ou $1$ para a base não normalizada, conforme a convenção).

Substituindo em $\nabla^2 \phi = -\rho/\varepsilon_0$:

$$
\sum_{n,m,l} \Phi_{nml}\,(-\Lambda_{nml})\,\psi_{nml}
= -\frac{1}{\varepsilon_0}\sum_{n,m,l} R_{nml}\,\psi_{nml}.
$$

Pela independência linear das autofunções, igualamos coeficiente a coeficiente:

$$
\boxed{
\Phi_{nml} = \frac{R_{nml}}{\varepsilon_0\,\Lambda_{nml}}
= \frac{4\,R_{nml}}{\varepsilon_0\,\pi^2(n^2+m^2+l^2)}
}
$$

O Laplaciano age diagonalmente na base de autofunções e o potencial é obtido dividindo o coeficiente de carga pelo autovalor correspondente.

### 2.4 Solução geral em série

$$
\boxed{
\phi(x,y,z) = \frac{1}{\varepsilon_0}\sum_{n,m,l=1}^{\infty}
\frac{4\,R_{nml}}{\pi^2(n^2+m^2+l^2)}
\sin\!\left(\frac{n\pi(x+1)}{2}\right)
\sin\!\left(\frac{m\pi(y+1)}{2}\right)
\sin\!\left(\frac{l\pi(z+1)}{2}\right)
}
$$

Que é o análogo 3D da série de Fourier senoidal para o problema eletrostático de Dirichlet no cubo $[-1,1]^3$.

## 3. Exemplo a ser Implementado

### 3.1 Densidade de carga

No código, escolhemos uma **fonte com descontinuidade suave na derivada (kink suave)**, construída a partir da função *softplus*:

$$
\rho(x,y,z) = A \cdot \text{softplus}\!\left(k(x - x_0)\right) \cdot \sin\!\left(\frac{\pi(y+1)}{2}\right) \cdot \sin\!\left(\frac{\pi(z+1)}{2}\right),
$$

com $A = 1{,}0 \times 10^{-6}$ C/m³, nitidez $k = 30$, transição em $x_0 = 0$, no domínio $[-1,1]^3$. A função softplus é definida por

$$
\text{softplus}(s) = \ln\!\left(1 + e^{s}\right),
$$

que é uma **aproximação suave e analítica da função de Heaviside** — vai a zero quando $s \to -\infty$, cresce linearmente quando $s \to +\infty$, e faz a transição suavemente em torno de $s = 0$ com escala $\Delta s \sim 1$.

Essa densidade é **suave e analítica** em todo o domínio, mas sua **derivada tem uma variação abrupta** em torno de $x = x_0$ — o que a torna muito mais severa do que a gaussiana do ponto de vista do método espectral. Não é autofunção do Laplaciano no cubo, então **não admite solução analítica fechada**; a integral de convolução com a função de Green não tem forma elementar.

Diferentemente da senoide, a softplus **não se anula nas faces** $x = \pm 1$: em $x = -1$, ela vale $\text{softplus}(-k) \approx e^{-k} \approx 10^{-13}$, desprezível; em $x = +1$, vale $\text{softplus}(k) \approx k = 30$. A densidade, portanto, **cresce monotonicamente em $x$**, com quase toda a carga concentrada na metade $x > 0$ do domínio. Nas direções $y$ e $z$, os senos garantem que $\rho$ se anula nas quatro faces laterais.

### 3.2 Potencial resultante

Como a fonte não é autofunção, o potencial é obtido como solução numérica do sistema linear

$$
\mathcal{L}\,\phi = f, \qquad f(\mathbf{x}) = -\frac{\rho(\mathbf{x})}{\varepsilon_0},
$$

com as linhas de fronteira substituídas por $\phi_i = 0$ (Dirichlet homogêneo). O resultado não tem forma fechada, mas herda as propriedades qualitativas esperadas:

- **Estrutura assimétrica em $x$**: o potencial é quase nulo em $x < 0$ (onde a carga é exponencialmente pequena) e cresce suavemente em direção a $x > 0$, com máximo próximo de $(1, 0, 0)$.
- **Modulação senoidal em $y$ e $z$**: os máximos locais ocorrem no plano central $y = 0$, $z = 0$, e o potencial decai para zero nas quatro faces laterais.
- **Gradiente mais acentuado em $x = 0$**: a região de transição do softplus deixa uma "impressão" no potencial — o gradiente $\partial_x \phi$ tem uma variação mais rápida ali do que em outras regiões, reflexo direto do kink na fonte.
- **Ausência de oscilações espúrias** para $N$ suficientemente grande, **mas com convergência mais lenta** do que nos casos suaves: o kink na derivada da fonte faz os coeficientes espectrais decaírem mais devagar, e para $N = 20$ pode haver oscilações visíveis perto de $x_0$.

A validação contra solução analítica não é possível, mas o **resíduo da equação de Poisson nos pontos interiores** continua sendo a métrica confiável: para a softplus com $k = 30$, espera-se resíduo em torno de $10^{-4}$ a $10^{-6}$ com $N = 20$ — **muito maior** do que os $10^{-14}$ do caso senoidal, justamente porque o kink está sub-resolvido. Aumentando $N$ para 40 ou 50, o resíduo cai exponencialmente, mas com taxa menor do que nos casos suaves.

### 3.3 Interpretação física

- **Fonte assimétrica com transição abrupta**: a softplus representa uma densidade de carga que é **praticamente nula em metade do domínio** ($x < 0$) e **cresce rapidamente em torno de $x = 0$**, saturando em $x > 0$. Fisicamente, é o análogo de uma **interface carregada** ou de uma **distribuição de carga em uma junção** — como uma camada de depleção em um semicondutor, ou uma transição de fase com acúmulo de carga em um lado.
- **Quebra de simetria em $x$**: diferentemente da gaussiana (que quebra simetria pela posição do centro) e da senoide (que preserva as simetrias do cubo), a softplus **quebra a simetria $x \to -x$** por construção, mesmo com centro em $x_0 = 0$. Isso acontece porque a função softplus **não é par nem ímpar** — ela é monotônica crescente.
- **Sinal**: a carga é **estritamente positiva em todo o domínio** (softplus é sempre positivo e os senos ao quadrado também), então o potencial também é positivo, com máximo na região de maior carga.
- **Amplitude**: com $A = 1{,}0 \times 10^{-6}$ C/m³, o potencial máximo está na ordem de **dezenas de volts**, dependendo do valor de $k$ e da geometria.
- **Relevância prática**: este é um caso **mais severo** que a gaussiana, porque testa a capacidade do método de lidar com **gradientes rápidos na fonte**. É o tipo de caso onde a diferença entre métodos espectrais (bem projetados) e diferenças finitas fica mais evidente.

### 3.4 Por que essa escolha é adequada

- **Força o uso do método numérico**: elimina qualquer possibilidade de solução analítica — a softplus não é autofunção, nem composição simples de autofunções.
- **Testa os limites do método espectral**: ao contrário da gaussiana (muito suave) e da senoide (analítica), a softplus tem uma **derivada com variação abrupta**, o que degrada a convergência espectral de exponencial pura para exponencial com taxa menor (e potencialmente algébrica se o kink for extremo). É o cenário onde a **escolha de $N$** e a **distribuição dos pontos GLC** importam mais.
- **Não muda a base de Chebyshev**: continua sendo $\{T_0, \dots, T_N\}$ em cada direção. O que muda é **quantos polinômios** são necessários para atingir precisão de máquina — com $k = 30$, $N \approx 20$ já dá resíduo pequeno, mas não no erro de máquina; para kink mais agudo ($k = 60$), seria necessário $N \gtrsim 40$; e para $k$ muito grande, a convergência passa a ser algébrica, com oscilações espúrias visíveis (fenômeno de Gibbs).
- **Prepara o pipeline para fontes arbitrárias com descontinuidade suave**: uma vez validado com a softplus, o pipeline está pronto para aceitar qualquer densidade definida no `parameters.yml` — incluindo fontes com rampas, transições, degraus suavizados e perfis de interface.

## 4. Cuidados ao Aplicar o Método Pseudo-Espectral

- **Mapeamento de domínio**:
Chebyshev é definido em $[-1,1]$; aplicamos em cada direção
$$
x_j = \frac{a_x+b_x}{2} + \frac{b_x-a_x}{2}\xi_j, \quad
y_k = \frac{a_y+b_y}{2} + \frac{b_y-a_y}{2}\xi_k, \quad
z_\ell = \frac{a_z+b_z}{2} + \frac{b_z-a_z}{2}\xi_\ell.
$$
Com $a=-1$ e $b=1$, o mapeamento reduz-se à identidade em cada direção, e os pontos de colocação coincidem com os pontos GLC de Chebyshev no domínio físico.

- **Matriz de diferenciação**:
Usamos $D_x^{(2)} = D_x \cdot D_x$ (produto matricial), e analogamente para $y$ e $z$, para preservar a precisão em $N$ moderado.

- **Operador Laplaciano discreto**:
O Laplaciano 3D em malha tensorial é construído via **soma de Kronecker**:
$$
\mathcal{L} = D_x^{(2)} \otimes I_y \otimes I_z + I_x \otimes D_y^{(2)} \otimes I_z + I_x \otimes I_y \otimes D_z^{(2)}.
$$
Isso evita construir manualmente matrizes 3D densas e preserva a estrutura tensorial.

- **Imposição das condições de contorno**:
Substituímos as linhas correspondentes aos pontos de fronteira do sistema $\mathcal{L}\phi = f$ pelas condições de Dirichlet ($\phi_i = 0$, faces aterradas em $x=\pm 1$, $y=\pm 1$, $z=\pm 1$). Isso destrói a estrutura espectral nessas linhas, mas é adequado para problemas suaves.

- **Pontos de colocação GLC**:
Em cada direção, há densidade maior nas bordas, o que é vantajoso para capturar gradientes próximos às faces do paralelepípedo.

- **Suavidade da fonte $\rho$**:
A convergência espectral exige $\rho$ analítica. A softplus é **analítica em todo o domínio** (a exponencial é $C^\infty$), então a convergência ainda é **exponencial** — mas com **taxa menor** do que nos casos senoidal e gaussiana, porque a derivada da softplus varia rapidamente em torno de $x = x_0$. Na prática, isso significa que para atingir o mesmo erro que a gaussiana atinge com $N = 20$, a softplus precisa de $N \approx 30$–$40$ (dependendo de $k$). Descontinuidades reais na densidade de carga degradariam a convergência para **algébrica** e gerariam oscilações espúrias.

- **Custo computacional**:
A matriz $\mathcal{L}$ tem tamanho $(N_x+1)(N_y+1)(N_z+1) \times \cdots$, ou seja, cresce rápido com $N$. Para $N=20$ em cada direção, são $\sim 9000$ incógnitas; para $N=40$, $\sim 69000$. O custo de LU denso é $\mathcal{O}(M^3)$ com $M$ o número total de incógnitas, o que mostra um rápido crescimento das operações a serem processadas.

## 5. Implicações para a Estrutura da Solução

### 5.1 Paridade nas faces (caso de domínio simétrico)

Se o domínio for simétrico em relação a um plano, por exemplo $x=0$, e $\rho$ for **par** em $x$, então:

- $\phi(\mathbf{x})$ é **par** em $x$: $\phi(x,y,z) = \phi(-x,y,z)$
- $\partial_x \phi$ é **ímpar** em $x$: $\partial_x \phi(0,y,z) = 0$
- A condição de regularidade em $x=0$ é naturalmente satisfeita por uma base com apenas potências pares em $x$

Nesse caso, usamos **apenas polinômios de Chebyshev de ordem par** $T_{2k}$ na direção $x$, o que melhora a convergência perto do plano de simetria.

**No caso da softplus**, a densidade **não é par nem ímpar** em $x$ — a função é monotônica crescente, quebrando simetria em $x \to -x$ **por construção**, mesmo com $x_0 = 0$. Nas direções $y$ e $z$, os senos são **ímpares** em relação aos planos centrais $y = 0$ e $z = 0$, mas como a densidade depende de $\sin(\cdots)$ (e não de $\sin^2$), a paridade em $y$ e $z$ **também é quebrada** pela presença do softplus em $x$. Portanto, usamos **toda a base** $T_0, \dots, T_N$ em cada direção.

### 5.2 Caso geral em $\Omega = [a_x,b_x]\times[a_y,b_y]\times[a_z,b_z]$

Como não há restrição de paridade, usamos toda a base $T_0, \dots, T_{N_x}$ em cada direção, e analogamente para $y$ e $z$.

### 5.3 Condições de contorno

Para condições de contorno de Dirichlet homogêneas ($\phi=0$ em $\partial\Omega$), as linhas da matriz correspondentes aos pontos de fronteira são substituídas por $\phi_i = 0$. Para condições de contorno de Dirichlet não-homogêneas, $\phi_i = V(\mathbf{x}_i)$ diferente de zero, o potencial pode ser prescrito nas faces (condutores mantidos a potenciais fixos).

## 6. Formulação Espectral

No código, resolvemos o mesmo problema da dedução analítica, mas em vez de expandir em autofunções analíticas (que só existem para geometrias simples e fontes específicas), expandimos em polinômios de Chebyshev e impomos as condições de contorno por substituição direta dos valores nas linhas da matriz.

| Elemento | Código espectral |
|---|---|
| Base de expansão | $T_k(x)T_l(y)T_m(z)$ |
| Operador| $\mathcal{L}$ (Kronecker de $D^{(2)}$) |
| Coeficientes do potencial| $\phi = \mathcal{L}^{-1} b$ |
| Condições de Contorno| impostas por substituição direta |
| Aplicabilidade| geometrias e fontes arbitrárias |

Para a **softplus** não existe expansão fechada em autofunções com apenas alguns modos — a função não é combinação de autofunções do Laplaciano. É exatamente por isso que o método espectral numérico se torna **indispensável**: ele resolve o mesmo tipo de problema sem depender da existência de uma base analítica conveniente. A **validação cruzada** com o caso senoidal (que tem gabarito) garante que o solver está correto; a partir daí, ele pode ser aplicado com confiança a fontes arbitrárias como a softplus — **inclusive aquelas com derivadas rapidamente variáveis**, que são o caso mais severo para métodos numéricos.

## 7. Tabela Resumo

| Aspecto | Equação de Poisson 3D em Eletrostática |
|---|---|
| Equação | $\nabla^2 \phi = -\rho/\varepsilon_0$ |
| Natureza | Linear, desacoplada |
| Singularidade no interior | Nenhuma para $\rho$ suave |
| Domínio | Fixo $\Omega = [-1,1]^3$ |
| Condições de contorno | Dirichlet ($\phi=0$, faces aterradas) |
| Densidade de carga | Kink suave: $A \cdot \text{softplus}\!\left(k(x-x_0)\right)\sin\!\left(\frac{\pi(y+1)}{2}\right)\sin\!\left(\frac{\pi(z+1)}{2}\right)$ |
| Solução analítica | **Não disponível** — fonte não é autofunção |
| Autofunções (para referência) | $\psi_{nml} = \sin\!\left(\frac{n\pi(x+1)}{2}\right)\sin\!\left(\frac{m\pi(y+1)}{2}\right)\sin\!\left(\frac{l\pi(z+1)}{2}\right)$ |
| Autovalores (para referência) | $\Lambda_{nml} = \frac{\pi^2}{4}(n^2+m^2+l^2)$ |
| Polinômios de Chebyshev | Expansão em cada direção, base completa $T_0, \dots, T_N$ |
| Convergência esperada | Exponencial com taxa **menor** que nos casos suaves |
| $N$ necessário (estimado) | $\sim 20$–$30$ para $k = 30$ |
| Métrica de validação | Resíduo interior $\|\mathcal{L}\phi - f\|_\infty$ |

## 8. Conclusão

A equação de Poisson 3D em eletrostática, com uma densidade de carga softplus, é o caso **mais severo** dos que testamos até agora:

1. **Sem singularidades:** a softplus é $C^\infty$ e analítica, então a convergência permanece exponencial.
2. **Domínio fixo:** sem autovalor de fronteira livre.
3. **Linear:** resolvemos com uma única fatoração LU.
4. **Estrutura tensorial:** o Laplaciano é construído por soma de Kronecker.
5. **Sem paridade explorável:** a softplus quebra simetria em todas as direções, então usamos a base completa $T_0, \dots, T_N$.
6. **Ausência de gabarito analítico:** sem fórmula fechada para $\phi$, a validação é pelo **resíduo da equação de Poisson** — que **não cai ao erro de máquina** como nos casos suaves, mas fica na faixa de $10^{-4}$ a $10^{-6}$ com $N = 20$, exigindo $N$ maior para precisão melhor.
7. **Teste de robustez:** ao contrário da gaussiana, a softplus tem **derivada rapidamente variável** em $x = x_0$, o que estressa os pontos GLC e a distribuição de Chebyshev. É exatamente o tipo de caso onde a escolha de $N$ e a filtragem espectral (se necessária) importam mais.

O pipeline mantém a mesma estrutura: monta o Laplaciano tensorial, impõe as condições de contorno de Dirichlet por substituição de linha, resolve o sistema linear denso e valida o resultado pelo resíduo interior. A diferença em relação ao caso senoidal é que o gabarito analítico é substituído pela **consistência numérica** (resíduo) e pela **comparação com o caso senoidal previamente validado** — estabelecendo a confiança necessária para usar o pipeline como gerador de dados de treinamento do MLP em situações onde a solução analítica é desconhecida **e a fonte tem estrutura abrupta**.